In [65]:
import os, sys

# Go up THREE levels (project root directory)
project_root = os.path.dirname(os.path.dirname(os.getcwd()))
project_root
# Append the new path to sys.path
if project_root not in sys.path:
    sys.path.append(project_root)
    print("Project root added to sys.path")
else:
    print("Project root already in sys.path")

Project root already in sys.path


In [66]:
# imports
import numpy as np
import pandas as pd
from utils import gro_processing

# Get data
df, title, num_atoms, box_dimensions = gro_processing.read_gro("../../data/npt-HK4.gro")
box_dimensions

array([11.24798, 11.24798, 11.24798])

In [67]:
# select_range = list(range(1,3 + 1))

# df = df[df["res_id"].isin(select_range)]

In [68]:
"""
If say I have 84 atoms,
and I want to select some to be the "electron clump"
then I would want to input a range or specific id of the atom,
so say like I want atom 24:56 and 80 and 84 to be my clump

Then from there I would want to parse the entire df, to only have data of the atoms of the respective id
and for every molecule as well
"""

num_atoms_one_mol = df.loc[df["res_id"] == 1].shape[0]
num_res = int(df.iloc[-1]["res_id"])

#UNCOMMENT FOLLOWING 2 LINES FOR USER INPUT
# print("Insert atom id for one molecule:")
# print("Use the following format: '20-30; 30; 20; 40-60' (for range use '-', for multi-input split using ';' ")
# user_input = input("Enter atom id:")
user_input = "15-20; 30; 31-33" # Test input, comment when not needed

#formatting user_input:
split_user_input = user_input.split("; ")
try:
    indices = [
        parts if len(parts := tuple(map(int, i.split("-")))) > 1 else int(i)
        for i in split_user_input
    ]  # not sure how to make this more readable but i like it in one line
except: print("Incorrect formatting, please follow the instructions above.")


new_df = pd.DataFrame()
for res in range(0,num_res):
    for i in indices:
        # in case range:
        if type(i) == tuple: 
            for id in range(i[0], i[1]+1):
                temp = df.loc[df["atom_id"] == (id + (num_atoms_one_mol * res))]
                new_df = pd.concat([new_df,temp])
        else: 
            temp = df.loc[df["atom_id"] == (i + (num_atoms_one_mol * res))]
            new_df = pd.concat([new_df,temp])        

In [69]:
select_res = new_df.loc[new_df["res_id"] == 2][['x','y','z']]
select_res

,x,y,z
98,0.089,0.684,11.229
99,0.296,0.703,11.186
100,0.427,0.676,11.233
101,0.508,0.726,11.177
102,0.455,0.598,0.096
103,0.560,0.587,0.125
113,0.419,0.449,0.396
114,0.523,0.477,0.380
115,0.277,0.743,11.046
116,0.361,0.716,10.937


In [70]:
# Finding centroid from atoms in new_df, honestly should combine with the above new_df for loop
from utils import oxygen_midpoints

def minimum_image_vector(p1, p2, box_length):
    dx = p1 - p2
    reciprocal_half_box = 2 / box_length
    return oxygen_midpoints.minimum_image_jit(dx, box_length, reciprocal_half_box)


centroid_df = pd.DataFrame()
for res in range(1, num_res + 1):
    select_res = new_df.loc[new_df["res_id"] == res][['x','y','z']]
    
    ref_atom = select_res.iloc[1].values
    vector_arr = np.empty((len(select_res),3))
    for atom in range(select_res.shape[0]):
        select_atom = select_res.iloc[atom].values
        vector = np.empty(3)
        for i in range(3):
            vector[i] = minimum_image_vector(ref_atom[i],select_atom[i], box_dimensions[0])
        vector_arr[atom] = vector
    
    midpoint = ref_atom + np.mean(vector_arr)
    temp_df = pd.DataFrame(columns= ['x','y','z'], data = midpoint[None,:], index=[res])
    centroid_df = pd.concat([centroid_df,temp_df])

centroid_df

,x,y,z
1,1.684533,0.485533,0.379533
2,0.255303,0.662303,11.145303
3,0.334500,10.782500,0.700500
4,0.492200,0.942200,0.266200
5,5.851638,11.292638,5.570638
...,...,...,...
1497,8.608800,7.095800,8.802800
1498,8.289200,0.958200,6.981200
1499,5.043700,0.641700,10.885700
1500,5.872700,10.656700,3.466700
